# Flight dataframe cleaning + Combining with Weather
Cleaning flight data obtained from web-scraper and combining with weather data

## 1. Imports & Setup

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import sqlite3
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

## 2. Examining Flights Data

In [11]:
final_df = pd.read_csv("departures_2024_all.csv")
print(final_df.info())
print(final_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3967209 entries, 0 to 3967208
Data columns (total 19 columns):
 #   Column                                    Dtype  
---  ------                                    -----  
 0   Carrier Code                              object 
 1   Date (MM/DD/YYYY)                         object 
 2   Flight Number                             float64
 3   Tail Number                               object 
 4   Destination Airport                       object 
 5   Scheduled departure time                  object 
 6   Actual departure time                     object 
 7   Scheduled elapsed time (Minutes)          float64
 8   Actual elapsed time (Minutes)             float64
 9   Departure delay (Minutes)                 float64
 10  Wheels-off time                           object 
 11  Taxi-Out time (Minutes)                   float64
 12  Delay Carrier (Minutes)                   float64
 13  Delay Weather (Minutes)                   float64
 14  De

## 3. Cleaning Flight Data

In [23]:
df = final_df.copy()

# Drop irrelevant columns
df = df.drop(columns=["Flight Number", "Tail Number", "Wheels-off time",
                       "Delay Carrier (Minutes)", "Delay National Aviation System (Minutes)",
                       "Delay Security (Minutes)", "Delay Late Aircraft Arrival (Minutes)"])

# Extract airport code from airport column
df["airport_code"] = df["airport"].str.extract(r'\((\w+)\)$')
df = df.drop(columns=["airport"])

# Parse date and extract useful features
df["Date (MM/DD/YYYY)"] = pd.to_datetime(df["Date (MM/DD/YYYY)"])
df["month"] = df["Date (MM/DD/YYYY)"].dt.month
df["day_of_week"] = df["Date (MM/DD/YYYY)"].dt.dayofweek
df = df.drop(columns=["Date (MM/DD/YYYY)"])

# Extract hour from scheduled departure time
df["scheduled_hour"] = pd.to_datetime(df["Scheduled departure time"], format="%H:%M").dt.hour
df = df.drop(columns=["Scheduled departure time", "Actual departure time"])

# Fill NaN delay values with 0
df["Delay Weather (Minutes)"] = df["Delay Weather (Minutes)"].fillna(0)
df["Departure delay (Minutes)"] = df["Departure delay (Minutes)"].fillna(0)

# Create target variable
df["weather_delayed"] = (df["Delay Weather (Minutes)"] > 0).astype(int)
df = df.drop(columns=["Delay Weather (Minutes)"])

# Drop remaining rows with NaN
df = df.dropna()

# Don't encode airport_code - keep it readable for the weather join
le = LabelEncoder()
df["Carrier Code"] = le.fit_transform(df["Carrier Code"])
df["airline"] = le.fit_transform(df["airline"])
# airport_code stays as a string (e.g. "EWR", "LAX") for now

print(df.info())
df

<class 'pandas.core.frame.DataFrame'>
Index: 3966862 entries, 0 to 3967207
Data columns (total 12 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   Carrier Code                      int64  
 1   Destination Airport               object 
 2   Scheduled elapsed time (Minutes)  float64
 3   Actual elapsed time (Minutes)     float64
 4   Departure delay (Minutes)         float64
 5   Taxi-Out time (Minutes)           float64
 6   airline                           int64  
 7   airport_code                      object 
 8   month                             float64
 9   day_of_week                       float64
 10  scheduled_hour                    float64
 11  weather_delayed                   int64  
dtypes: float64(7), int64(3), object(2)
memory usage: 393.4+ MB
None


,Carrier Code,Destination Airport,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Taxi-Out time (Minutes),airline,airport_code,month,day_of_week,scheduled_hour,weather_delayed
0,1,DTW,110.0,92.0,12.0,12.0,1,EWR,1.0,0.0,13.0,0
1,1,SLC,325.0,291.0,-5.0,23.0,1,EWR,1.0,0.0,7.0,0
2,1,MSP,184.0,152.0,-10.0,13.0,1,EWR,1.0,0.0,12.0,0
3,1,MSP,194.0,161.0,4.0,20.0,1,EWR,1.0,0.0,18.0,0
4,1,SLC,330.0,278.0,18.0,17.0,1,EWR,1.0,0.0,18.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
3967203,2,DEN,105.0,110.0,-6.0,27.0,3,ICT,12.0,6.0,9.0,0
3967204,2,ORD,120.0,101.0,161.0,9.0,3,ICT,12.0,0.0,7.0,0
3967205,2,DEN,105.0,152.0,-2.0,71.0,3,ICT,12.0,0.0,9.0,0
3967206,2,ORD,120.0,128.0,-2.0,15.0,3,ICT,12.0,1.0,7.0,0


## 4. Examining Weather Data

In [30]:
conn = sqlite3.connect("weather.db")
locations_df = pd.read_sql("SELECT * FROM locations", conn)
details_df = pd.read_sql("SELECT * FROM details", conn)
weather_df = pd.merge(details_df, locations_df, on=["EPISODE_ID", "EVENT_ID"])
print(weather_df.shape)
weather_df.head()


(41603, 22)


,EPISODE_ID,EVENT_ID,STATE,YEAR,MONTH_NAME,EVENT_TYPE,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,BEGIN_LAT,...,END_LON,YEARMONTH,LOCATION_INDEX,RANGE,AZIMUTH,LOCATION,LATITUDE,LONGITUDE,LAT2,LON2
0,197543,1221942,OREGON,2024,November,Funnel Cloud,14-NOV-24 12:15:00,PST-8,14-NOV-24 12:16:00,44.324,...,-123.3104,202411,1,0.73,WNW,MONROE,44.3240,-123.3136,4419440,12318816
1,197543,1221942,OREGON,2024,November,Funnel Cloud,14-NOV-24 12:15:00,PST-8,14-NOV-24 12:16:00,44.324,...,-123.3104,202411,2,0.68,NW,MONROE,44.3264,-123.3104,4419584,12318624
2,197549,1221959,OREGON,2024,November,Heavy Rain,20-NOV-24 08:00:00,PST-8,21-NOV-24 08:00:00,45.620,...,-123.4283,202411,1,15.10,W,BANKS,45.6200,-123.4325,4537200,12325950
3,197549,1221959,OREGON,2024,November,Heavy Rain,20-NOV-24 08:00:00,PST-8,21-NOV-24 08:00:00,45.620,...,-123.4283,202411,2,14.90,W,BANKS,45.6200,-123.4283,4537200,12325698
4,196653,1216653,IOWA,2024,November,Tornado,05-NOV-24 11:01:00,CST-6,05-NOV-24 11:02:00,40.803,...,-92.3437,202411,1,3.94,ESE,BELKNAP,40.8030,-92.3482,4048180,9220892


In [26]:
print(locations_df.columns.tolist())
print(details_df.columns.tolist())

['YEARMONTH', 'EPISODE_ID', 'EVENT_ID', 'LOCATION_INDEX', 'RANGE', 'AZIMUTH', 'LOCATION', 'LATITUDE', 'LONGITUDE', 'LAT2', 'LON2']
['EPISODE_ID', 'EVENT_ID', 'STATE', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON']


In [28]:
print(details_df["BEGIN_DATE_TIME"].min())
print(details_df["BEGIN_DATE_TIME"].max())

01-APR-24 00:00:00
31-OCT-24 22:47:00
